# Latent Bayesian Hyperparameter Optimization

Select one dataset, define a mixed hyperparameter search space, and optimize SCALP graph parameters with Bayesian optimization in a low-dimensional latent space.

In [ ]:
import sys
import subprocess


def ensure_bo_dependencies():
    try:
        import botorch  # noqa: F401
        import gpytorch  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", "..[bo]"])


ensure_bo_dependencies()


In [ ]:
import numpy as np

from scalp_lite.metrics import score_embedding
from scalp_lite.notebook_utils import dataset_config, load_preprocessed_data, make_estimator
from scalp_lite.optimization import latent_bayesopt


selected_dataset = "pancreas"
dataset = dataset_config(selected_dataset)

# Keep objective evaluations manageable. Increase after validating the search setup.
preprocess_overrides = {"max_cells": 2000}
random_state = 0
estimator = make_estimator(dataset, n_components=100, random_state=random_state)
adata = load_preprocessed_data(estimator, dataset, **preprocess_overrides)
adata


In [ ]:
base_graph_params = {
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 2,
    "metric": "euclidean",
    "assignment_quantile": 0.8,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "binary",
    "mutual_neighbors": False,
    "neighbor_mode": "distance",
    "symmetrize": True,
}

search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 40]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 8]},
    "assignment_quantile": {"type": "float", "bounds": [0.05, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 30]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

base_graph_params, search_space


In [ ]:
def scalp_objective(params):
    graph_params = {**base_graph_params, **params}
    trial = adata.copy()
    trial.obsm["X_scalp"] = estimator.embed(
        trial,
        **graph_params,
        embedding_random_state=random_state,
        n_epochs=100,
    )
    scores = score_embedding(
        trial,
        embedding_key="X_scalp",
        batch_key=dataset["batch_key"],
        label_key=dataset["label_key"] if dataset["label_key"] in trial.obs else None,
    ).iloc[0]
    # Maximize biological coherence and batch mixing; tune weights for your target use case.
    return float(scores["knn_label_agreement"] + 0.25 * scores["batch_mixing"] - 0.05 * scores["graph_density"])


In [ ]:
# Start with PCA for a cheap sanity check. Switch to "gplvm" for nonlinear latent optimization.
bo_result = latent_bayesopt(
    scalp_objective,
    search_space,
    n_initial=6,
    latent_dim=3,
    n_iterations=4,
    embedding_model="pca",
    acquisition="ei",
    batch_size=1,
    random_state=random_state,
)

bo_result["best_params"], bo_result["best_score"]


In [ ]:
bo_result["history"].sort_values("score", ascending=False).head(10)
